# Data exploration

Inspect images, annotations, class balance, image dimensions, corrupt files, and official splits. Decide classification or segmentation and an evaluation protocol before training.

Setup only. Implementation will be added one learning step at a time.

In [1]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "configs" / "experiment_config.yaml").exists(), "Open this notebook from the project folder."
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", device)


Project: /Users/farabi/Farabi/ML-Projects/CV/Oxford-IIT Pet
Python: /Users/farabi/Farabi/ML-Projects/CV/Oxford-IIT Pet/.venv/bin/python
PyTorch: 2.14.1
Device: mps


In [6]:
from datasets import load_dataset

dataset = load_dataset("timm/oxford-iiit-pet")

In [5]:
from tqdm.auto import tqdm

In [11]:
#Checking the structure of the dataset,what labels are there and what splits it has ,
#It doesnt have validation split, it only has test and train dataset


print(dataset)
print(dataset.keys)
split_name = next(iter(dataset.keys()))
print(dataset[split_name].features)


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3680
    })
    test: Dataset({
        features: ['image', 'label', 'image_id', 'label_cat_dog'],
        num_rows: 3669
    })
})
<built-in method keys of DatasetDict object at 0x12179d390>
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['abyssinian', 'american_bulldog', 'american_pit_bull_terrier', 'basset_hound', 'beagle', 'bengal', 'birman', 'bombay', 'boxer', 'british_shorthair', 'chihuahua', 'egyptian_mau', 'english_cocker_spaniel', 'english_setter', 'german_shorthaired', 'great_pyrenees', 'havanese', 'japanese_chin', 'keeshond', 'leonberger', 'maine_coon', 'miniature_pinscher', 'newfoundland', 'persian', 'pomeranian', 'pug', 'ragdoll', 'russian_blue', 'saint_bernard', 'samoyed', 'scottish_terrier', 'shiba_inu', 'siamese', 'sphynx', 'staffordshire_bull_terrier', 'wheaten_terrier', 'yorkshire_terrier']), 'image_id': Value('string'), 'label_ca

In [12]:
sample = dataset[split_name][0]
print(sample.keys())

dict_keys(['image', 'label', 'image_id', 'label_cat_dog'])


In [13]:
'''
We do not have validation ds, so we will create a internal validation ds from the train set
we will use internal test set to fine tune our model's weights and run experiments
'''

split = dataset["train"].train_test_split(
    test_size = 0.2,
    seed = 42,
    stratify_by_column="label"
)


train_ds = split["train"]
val_ds = split["test"]
test_ds = dataset["test"]

print("Training:", len(train_ds))
print("Validation:", len(val_ds))
print("Testing:", len(test_ds))

Training: 2944
Validation: 736
Testing: 3669


In [ ]:
from torchvision import transforms

#transform the train_data to do data augmentation which will help the model in future to generalize 


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size = 224,
        scale = (0.8,1.0)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], #from ImageNet statistics
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = val_transform


In [16]:
import torch
from torch.utils.data import Dataset,DataLoader

class PetDataset(Dataset):
    def __init__(self,hf_dataset,transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self,idx):
        sample = self.dataset[idx]

        image = sample["image"].convert("RGB")
        label = sample["label"]

        if self.transform is not None:
            image = self.transform(image)

        return image,label

In [18]:
train_dataset = PetDataset(train_ds,train_transform)
val_dataset = PetDataset(val_ds,val_transform)
test_dataset = PetDataset(test_ds,test_transform)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

2944
736
3669


In [19]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)


In [20]:

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)
print("First 5 labels:", labels[:5])


Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64
First 5 labels: tensor([26, 12, 11, 35,  7])


In [21]:
import torch
import torch.nn as nn

In [27]:
class SimpleCNN(nn.Module):
    def __init__(self,num_classes = 37):
        super().__init__()

        self.features = nn.Sequential(
            #Block 1
            nn.Conv2d(3,16,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #Block2
            nn.Conv2d(16,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            #Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Linear(64,num_classes)

    def forward(self, x):
            x = self.features(x)
            x = self.pool(x)
            x = torch.flatten(x, start_dim=1)
            x = self.classifier(x)

            return x

In [ ]:

import torch
import torch.nn as nn
import copy

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

model = SimpleCNN(num_classes=37).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

num_epochs = 10


Using device: mps


In [ ]:

import torch
import copy

def fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=10,
    head_only=False,
    freeze_bn=False
):
    model = model.to(device)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_acc = -1.0
    best_state = None

    for epoch in range(epochs):

        # ============== TRAINING ==============
        if head_only:
            model.eval()  # Keep frozen backbone BN statistics fixed
            model.get_classifier().train()
        else:
            model.train()

        if freeze_bn:
            for module in model.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()


        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * labels.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # ============== VALIDATION ==============
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.inference_mode():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * labels.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        # ============== HISTORY ==============
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # ============== BEST MODEL ==============
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc*100:.2f}%"
        )

    # Restore best validation checkpoint
    model.load_state_dict(best_state)

    print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

    return history


In [ ]:

import matplotlib.pyplot as plt

def plot_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="Train")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["train_acc"], label="Train")
    axes[1].plot(epochs, history["val_acc"], label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [50]:
'''
import timm

model = timm.create_model(
    "resnet18",
    pretrained=True,
    num_classes = 37
)

model.requires_grad_(False)

model.get_classifier().requires_grad_(True)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.get_classifier().parameters(),
    lr=0.001,
    weight_decay=1e-4
)

model_headTuned = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=loss_fn,
    device=device,
    epochs=10,
    head_only=True
)

'''

'\nimport timm\n\nmodel = timm.create_model(\n    "resnet18",\n    pretrained=True,\n    num_classes = 37\n)\n\nmodel.requires_grad_(False)\n\nmodel.get_classifier().requires_grad_(True)\n\nloss_fn = nn.CrossEntropyLoss()\n\noptimizer = torch.optim.AdamW(\n    model.get_classifier().parameters(),\n    lr=0.001,\n    weight_decay=1e-4\n)\n\nmodel_headTuned = fit(\n    model=model,\n    train_loader=train_loader,\n    val_loader=val_loader,\n    optimizer=optimizer,\n    criterion=loss_fn,\n    device=device,\n    epochs=10,\n    head_only=True\n)\n\n'

In [37]:
model = model.to(device)

In [40]:
head_only_state = {
    k: v.detach().cpu().clone()
    for k, v in model.state_dict().items()
}

In [41]:
torch.save(
    head_only_state,
    "resnet18_head_only_best.pth"
)

In [43]:
model.requires_grad_(True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 1e-5,
    weight_decay=1e-4
)



In [44]:

model_resnet18_full = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=5,
    head_only=False
)


Epoch [1/5] | Train Loss: 1.0949 | Train Acc: 73.71% | Val Loss: 0.9182 | Val Acc: 78.40%
Epoch [2/5] | Train Loss: 0.9468 | Train Acc: 78.16% | Val Loss: 0.8461 | Val Acc: 79.76%
Epoch [3/5] | Train Loss: 0.8289 | Train Acc: 81.96% | Val Loss: 0.7534 | Val Acc: 82.07%
Epoch [4/5] | Train Loss: 0.7342 | Train Acc: 84.10% | Val Loss: 0.6615 | Val Acc: 84.24%
Epoch [5/5] | Train Loss: 0.6566 | Train Acc: 85.90% | Val Loss: 0.6239 | Val Acc: 85.33%

Best Validation Accuracy: 85.33%


In [45]:

checkpoint = torch.load(
    "resnet18_head_only_best.pth",
    map_location="cpu",
    weights_only=True
)

print("Number of state entries:", len(checkpoint))
print("Classifier shape:", checkpoint["fc.weight"].shape)


Number of state entries: 122
Classifier shape: torch.Size([37, 512])


In [47]:
checkpoint_path = (
    "../kaggle/checkpoint-dataset/"
    "resnet18_head_only_best.pth"
)

# Recreate the model architecture
model = timm.create_model(
    "resnet18",
    pretrained=False,
    num_classes=37
)

# Load your previously trained weights
state_dict = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

In [49]:
model.requires_grad_(False)

model.layer4.requires_grad_(True)

model.get_classifier().requires_grad_(True)


# Verify trainable parameters
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

Total parameters: 11,195,493
Trainable parameters: 8,412,709
Frozen parameters: 2,782,784


In [54]:
import torch
import timm


In [52]:
from pathlib import Path
from sklearn.metrics import accuracy_score,f1_score

project_root = Path.cwd()

if not (project_root / "outputs").exists():
    project_root = project_root.parent

checkpoints = {
    "Head-only": (
        project_root / "kaggle/checkpoint-dataset/"
        "resnet18_head_only_best.pth"
    ),
    "Layer4 + Head": (
        project_root / "outputs/"
        "resnet18_layer4_best.pth"
    )
}

In [58]:
def evaluate_checkpoint(checkpoint_path, test_loader):
    model = timm.create_model(
        "resnet18",
        pretrained=False,
        num_classes=37
    )

    state_dict = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()

    y_true = []
    y_pred = []

    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(device)

            logits = model(images)
            predictions = logits.argmax(dim=1)

            y_true.extend(labels.tolist())
            y_pred.extend(predictions.cpu().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")

    return accuracy, macro_f1, y_true, y_pred

In [55]:

for name, checkpoint_path in checkpoints.items():
    accuracy, macro_f1 = evaluate_checkpoint(
        checkpoint_path, test_loader
    )

    print(f"\n{name}")
    print(f"Test Accuracy: {accuracy:.4%}")
    print(f"Macro F1: {macro_f1:.4f}")


Head-only
Test Accuracy: 89.5339%
Macro F1: 0.8939

Layer4 + Head
Test Accuracy: 89.8610%
Macro F1: 0.8973


In [59]:

import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

accuracy, macro_f1, y_true, y_pred = evaluate_checkpoint(
    checkpoints["Layer4 + Head"],
    test_loader
)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=list(range(37)),
    zero_division=0
)

f1_results = pd.DataFrame({
    "Class ID": range(37),
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
})

print("Test Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nLowest F1 classes:")
print(f1_results.sort_values("F1").head(10))

print("\nHighest F1 classes:")
print(f1_results.sort_values("F1", ascending=False).head(10))


Test Accuracy: 0.8986
Macro F1: 0.8973

Lowest F1 classes:
    Class ID  Precision    Recall        F1  Support
34        34   0.635294  0.606742  0.620690       89
2          2   0.783784  0.580000  0.666667      100
26        26   0.704762  0.740000  0.721951      100
1          1   0.711864  0.840000  0.770642      100
9          9   0.855422  0.710000  0.775956      100
27        27   0.853933  0.760000  0.804233      100
5          5   0.735537  0.890000  0.805430      100
6          6   0.765766  0.850000  0.805687      100
20        20   0.840426  0.790000  0.814433      100
8          8   0.767241  0.898990  0.827907       99

Highest F1 classes:
    Class ID  Precision    Recall        F1  Support
18        18   0.989899  0.989899  0.989899       99
15        15   0.980198  0.990000  0.985075      100
17        17   0.980198  0.990000  0.985075      100
25        25   0.989899  0.980000  0.984925      100
22        22   0.989899  0.980000  0.984925      100
19        19   1.00